In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
import os
import pickle
import pandas as pd
import numpy as np
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random
from itertools import product
from typing import List, Tuple, Optional
from IPython.display import clear_output
%matplotlib inline

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

# desktop version
data_folder = "G:\Meu Drive\mestrado_final_files\data\\"
models_folder = "G:\Meu Drive\mestrado_final_files\models\\SpentTimeModel_MultiTowers\\"

# google colab version
# data_folder = "drive/MyDrive/mestrado_final_files/data/"
# models_folder = "drive/MyDrive/mestrado_final_files/models/SpentTimeModel_MultiTowers/"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
device

device(type='cpu')

In [4]:
print(f"Pandas version: {pd.__version__},\nNumpy Version: {np.__version__},\
      \nTorch Version: {torch.__version__},\nMatplotlib Version: {plt.matplotlib.__version__}")

Pandas version: 2.2.3,
Numpy Version: 2.2.6,      
Torch Version: 2.8.0+cpu,
Matplotlib Version: 3.10.0


In [5]:
pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", False)

In [6]:
BATCH_SIZE = 1024
EPOCHS = 50
VOCAB_SIZE = 601
TIME_SIZE = 301

run_models = 1

In [8]:
%run 01_functions_training.ipynb
#run drive/MyDrive/mestrado_final_files/notebooks/01_functions_training.ipynb

In [9]:
with open(data_folder + 'data_time_features.pkl', 'rb') as f:
    df_time_features = pickle. load(f)

In [10]:
with open(data_folder + 'dict_set_covariables.pkl', 'rb') as f:
    dict_set_covariables = pickle.load(f)

In [ ]:
list_seasons = ['2018-19', '2019-20', '2020-21', '2021-22', '2022-23']
list_context = [5]
list_embedding_layer_dim = [32, 64]
list_num_neurons = [50]
list_num_layers = [3, 5]
list_dropout = [0, 0.3]
list_layernorm = [False, True]

In [16]:
model_mask_name_total = "{}_seasondata{}_context{}_embedding{}_numlayers{}_numneurons{}_{}_layernorm{}_setcovariables56_parameters.pth"

In [17]:
all_combinations = list(product(list_seasons, list_context, list_embedding_layer_dim, list_num_neurons, list_num_layers,
                                list_dropout, list_layernorm))

In [18]:
len(all_combinations)

40

In [19]:
i = 0
for comb in all_combinations:
    i += 1
    print(i)
    season, CONTEXT_SIZE, EMBEDDING_LAYER_SIZE, NUM_HIDDEN_NEURONS, NUM_LAYERS, dropout_rate, layer_norm = comb

    print(comb)
    index_num_covariables1 = dict_set_covariables["index"][5]
    num_covariables1 = dict_set_covariables["covariables"][5]

    index_num_covariables2 = dict_set_covariables["index"][6]
    num_covariables2 = dict_set_covariables["covariables"][6]


    # ================== DADOS TREINO/DEV - MODELO TOTAL ==================
    df_ = df_time_features.query("season == @season & season_split == 'train'")

    trn_tkn = torch.tensor(np.asarray(df_["token_features"].to_list()), dtype = torch.int64, device = device)
    trn_num = torch.tensor(np.asarray(df_["numerical_features"].to_list()), dtype = torch.float32, device = device)
    trn_y   = torch.tensor(np.asarray(df_["target_time"].to_list()), dtype = torch.int64, device = device)
    trn_mask = torch.tensor(build_logit_mask_spent_time_models(df_), dtype = torch.bool, device = device)

    df_ = df_time_features.query("season == @season & season_split == 'dev'")

    dev_tkn = torch.tensor(np.asarray(df_["token_features"].to_list()), dtype = torch.int64, device = device)
    dev_num = torch.tensor(np.asarray(df_["numerical_features"].to_list()), dtype = torch.float32, device = device)
    dev_y   = torch.tensor(np.asarray(df_["target_time"].to_list()), dtype = torch.int64, device = device)
    dev_mask = torch.tensor(build_logit_mask_spent_time_models(df_), dtype = torch.bool, device = device)

    del df_
    # =======================================================================

    tp_dropout = "nodropout" if dropout_rate == 0 else "dropout" + str(dropout_rate).replace(".", "")

    time_model = SpentTimeModel_MultiTowers(
        context_size = CONTEXT_SIZE,
        vocab_size = VOCAB_SIZE,
        time_size = TIME_SIZE,
        embedding_dim = EMBEDDING_LAYER_SIZE,
        numeric_input_dims = [len(index_num_covariables1), len(index_num_covariables2)],
        token_tower_dim    = 32,
        numeric_tower_dims = 16,
        num_hidden_neurons = NUM_HIDDEN_NEURONS,
        num_layers         = NUM_LAYERS,
        activation_function = nn.GELU,
        dropout_rate = dropout_rate,
        use_layer_norm = layer_norm)

    str_season = season.replace("-", "")

    model_filename = model_mask_name_total\
    .format(time_model.__class__.__name__, str_season, CONTEXT_SIZE, EMBEDDING_LAYER_SIZE, NUM_LAYERS, NUM_HIDDEN_NEURONS, tp_dropout, str(layer_norm))

    results_filename = "results_"  + model_filename.replace("_parameters.pth", ".pkl")

    if results_filename in os.listdir(models_folder):
        print("Already fitted.")
        continue

    # ================== TREINO MODELO TOTAL ==================

    time_model, trnloss, devloss = train_spent_time_model_multi_towers(
        model = time_model,
        train_token_data = trn_tkn[:, -CONTEXT_SIZE:],
        train_numerical_groups = [trn_num[:, index_num_covariables1], trn_num[:, index_num_covariables2]],
        train_targets = trn_y,
        train_mask_data = trn_mask,
        batch_size = BATCH_SIZE,
        n_epochs = EPOCHS,
        lr = 0.002,
        val_token_data = dev_tkn[:, -CONTEXT_SIZE:],
        val_numerical_groups = [dev_num[:, index_num_covariables1], dev_num[:, index_num_covariables2]],
        val_targets = dev_y,
        val_mask_data = dev_mask,
        params_optimizer = {'factor': 0.5, 'threshold': 0, 'patience': 1},
        device = device)

    df_importance = permutation_importance_ngram_multi_towers(
        model = time_model,
        target = dev_y,
        tokens = dev_tkn[:, -CONTEXT_SIZE:],
        numerical_groups = [dev_num[:, index_num_covariables1], dev_num[:, index_num_covariables2]],
        names_groups = [num_covariables1, num_covariables2],
        num_repeats = 10,
        mask_data = dev_mask)

    print(df_importance)

    output = time_model(trn_tkn[:, -CONTEXT_SIZE:], [trn_num[:, index_num_covariables1], trn_num[:, index_num_covariables2]])
    output = output.masked_fill(trn_mask, -1e9)    
    baseline_loss = F.cross_entropy(output, trn_y).item()

    print(f"train data loss: {baseline_loss}")

    torch.save(time_model.state_dict(), models_folder + model_filename)

    dict_results = {
        "class_model": time_model.__class__.__name__,
        "model_filename": model_filename,
        "token_context_size": CONTEXT_SIZE,
        "token_embedding_dim": EMBEDDING_LAYER_SIZE,
        "training_epochs": EPOCHS,
        "numerical_covariables": [num_covariables1, num_covariables2],
        "token_tower_dim": 32,
        "numeric_tower_dims": 16,
        "training_batch_loss": np.asarray(trnloss),
        "train_loss": baseline_loss,
        "dev_loss": np.asarray(devloss),
        "num_parameters": get_num_parameters(time_model),
        "variable_importance": df_importance}

    with open(models_folder + results_filename, 'wb') as f:
        pickle.dump(dict_results, f)

    clear_output(wait = True)

13
('2019-20', 5, 32, 50, 5, 0, False)


KeyboardInterrupt: 